# MeetingMind — turn meeting transcripts into decisions, owners and searchable team memory

### A complete project in one hour · ChromaDB + LangChain + LangGraph + LangSmith

---

## The problem (this is the "impactful" part)

> Every team records meetings. Almost no team can answer *"what did we decide about pricing three weeks ago?"* or *"what am I actually on the hook for?"* without a human scrolling through transcripts.
>
> Decisions get made, then lost. Action items get agreed, then orphaned.

**MeetingMind** does three things:

1. **Extracts** structured notes from raw transcripts — decisions, action items with an owner and a due date, and risks.
2. **Produces an action register** — a real table you could paste into a stand-up.
3. **Answers questions** about the team's history: *"what did we decide about the launch date?"* and *"what does Priya owe?"*

That third point hides the key design lesson, which is what makes this project worth an hour of your time.

## The one design idea that matters 📌

Those two questions look similar and need **completely different machinery**:

| Question | What it really is | Mechanism |
|---|---|---|
| *"What did we decide about pricing?"* | a **meaning** question — the word "pricing" may not appear | **vector search** (embeddings) |
| *"What does Priya owe?"* | a **filter** question — an exact match on an owner field | **metadata query, no embeddings at all** |

Running the second one through vector search gives you *some* of Priya's items — whichever happen to be semantically nearest. That is a wrong answer that looks right.

Chroma handles both, and that is why we use it here: `similarity_search()` for meaning, `.get(where=…)` for exact filters — **the second doesn't even call the embedding model.**

## Why Chroma for this

| | Chroma | FAISS |
|---|---|---|
| Persistence | built in: `persist_directory=` and it's on disk | you call `save_local()` / `load_local()` yourself |
| Metadata filters | rich operators: `$eq`, `$gte`, `$in`, `$and` | dict equality or a Python callable |
| Metadata-only query | ✅ `.get(where=…)` — **no vector search** | ❌ you must search first |
| Setup | `pip install`, no server | `pip install`, no server |

For a project built around "filter by owner, project and week", Chroma's query language is the right tool.

## The hour

| Minutes | Section | What you get |
|---|---|---|
| 0–5 | §0 Setup | keys and a green preflight |
| 5–10 | §1 Data | 6 realistic transcripts |
| 10–25 | §2–3 LangChain | structured extraction + an action register |
| 25–40 | §4 Chroma | a persistent, filterable memory |
| 40–55 | §5–6 LangGraph | the 3-way router and a working assistant |
| 55–60 | §7–8 LangSmith | tracing, 4 metrics, a scorecard |

## Contents

| § | Topic |
|---|---|
| 0 | Setup |
| 1 | The transcripts |
| 2 | LangChain: structured extraction |
| 3 | The action register (a real deliverable) |
| 4 | ChromaDB: persistent, filterable memory |
| 5 | LangGraph: state and the router |
| 6 | Running the assistant |
| 7 | LangSmith: tracing |
| 8 | LangSmith: 4 metrics and a scorecard |
| 9 | Self-check, next steps, resume bullet |

## 0. Setup  ⏱ 5 min

⚠️ **No version pins.** Pinning two packages of the same family to guessed versions makes pip fail resolution — and pip is all-or-nothing, so *nothing* installs and every later cell dies with `ModuleNotFoundError`.

In [ ]:
# %pip is a Jupyter magic: it installs into the SAME Python that runs this notebook.
# Plain !pip can install into a different interpreter - the #1 cause of
# "I installed it but I still get ModuleNotFoundError".
#   -q = quiet    -U = upgrade if an older version is present
%pip install -q -U langchain langchain-openai langchain-chroma chromadb langgraph langsmith pandas

print("Install finished.")
print("If a 'RESTART SESSION' button appeared, click it, then continue from the next cell.")

In [ ]:
import os          # os.environ = this process's environment variables
import getpass     # getpass = input that is NOT echoed to screen, like a password field

# An environment variable, not a variable in code, because:
#   1. every LangChain integration reads OPENAI_API_KEY from there automatically
#   2. env vars behave identically on your laptop, in Docker and on a server
#   3. nothing gets saved into the .ipynb, so the notebook is safe to share
if not os.environ.get("OPENAI_API_KEY"):        # .get() returns None instead of raising
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (sk-...): ")

# Define models ONCE at the top. Changing model later = editing one line, not fifty.
CHAT_MODEL  = "gpt-4o-mini"                # cheap, fast, plenty for this
EMBED_MODEL = "text-embedding-3-small"     # a DIFFERENT model: it makes vectors, not text

CHROMA_DIR  = "./meetingmind_db"           # where Chroma keeps its files on disk
COLLECTION  = "meeting_memory"             # ⚠️ Chroma requires 3-512 chars, [a-zA-Z0-9._-]

print("Key set |", CHAT_MODEL, "|", EMBED_MODEL)

In [ ]:
# PREFLIGHT: catch problems HERE, at cell 3, instead of at cell 30.
problems = []
try:
    import pandas as pd
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_chroma import Chroma
    from langgraph.graph import StateGraph
    from langsmith import traceable
except ImportError as e:
    problems.append(str(e))

if problems:
    print("PROBLEMS:", problems)
    print("-> re-run the install cell, RESTART the session, then run this cell again.")
else:
    # temperature=0 -> the same input gives the same output. Essential for extraction,
    # and it is what makes debugging possible at all.
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, timeout=45, max_retries=2)
    emb = OpenAIEmbeddings(model=EMBED_MODEL)
    print("live test ->", llm.invoke("Reply with the single word: ready").content)
    print("ALL GOOD.")

In [ ]:
# Everything the project imports, in one place.
import time, json, textwrap
from typing import TypedDict, Annotated, Literal, Optional, List

# --- LangChain ---
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document

# --- pydantic: describes the SHAPE of data we want back from the model ---
from pydantic import BaseModel, Field

# --- LangGraph ---
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# --- LangSmith ---
from langsmith import traceable

print("imports ready")

## 1. The transcripts  ⏱ 5 min

Six short meeting transcripts across two projects and four weeks. They are written to be realistically messy: people interrupt, decisions are buried mid-sentence, and owners are named casually ("Priya, can you…").

They're written to disk because real projects read files — and because the loading code you learn here is the code you'll actually ship.

In [ ]:
os.makedirs("transcripts", exist_ok=True)     # exist_ok=True -> don't crash if it exists

TRANSCRIPTS = {
    "M-01": {
        "project": "Atlas", "week": 11, "date": "2025-03-12", "title": "Atlas weekly sync",
        "text": """
        Sam: Right, Atlas. Where are we on the checkout rewrite?
        Priya: Backend is done. Frontend is maybe 70%. The blocker is the payments sandbox,
          it's been flaky all week.
        Sam: How flaky?
        Priya: Two full outages. We lost about a day and a half.
        Dan: I can raise it with the vendor, I have a call with them Thursday anyway.
        Sam: Do that. Let's also make a call on the launch date - I don't think 28 March is
          realistic any more.
        Priya: Agreed. If we slip to 11 April we get a proper QA week.
        Sam: Fine, we're moving Atlas launch to 11 April. I'll tell the exec team.
        Dan: One more thing, we still haven't decided whether to support Apple Pay at launch.
        Sam: Let's cut it from v1. We can add it in May. Priya, can you write up the
          frontend risk list before Friday?
        Priya: Will do.
        """,
    },
    "M-02": {
        "project": "Beacon", "week": 11, "date": "2025-03-13", "title": "Beacon kickoff",
        "text": """
        Maya: This is the Beacon kickoff. Goal is a self-serve onboarding flow by end of Q2.
        Dan: What's the scope? Is this just signup, or the whole first-week experience?
        Maya: Signup plus the first three activation emails. Nothing beyond that.
        Sam: Who's engineering on this?
        Maya: Two people, Dan and one hire we haven't made yet.
        Sam: So the hire is on the critical path. That's a risk.
        Maya: It is. We've decided to start with a contractor while hiring runs in parallel,
          so we're not blocked.
        Dan: I'd want a decision on the email vendor before I build anything.
        Maya: Let's standardise on the one Atlas already uses - no point running two.
        Dan: Works for me.
        Maya: Dan, can you produce a technical design doc by 21 March? And I'll own the
          contractor search.
        """,
    },
    "M-03": {
        "project": "Atlas", "week": 12, "date": "2025-03-19", "title": "Atlas weekly sync",
        "text": """
        Sam: Payments sandbox - resolved?
        Dan: Yes. Vendor moved us to a dedicated sandbox instance. No outages since Monday.
        Sam: Good. Priya, frontend?
        Priya: 90%. The risk list is done, I shared it Friday. Main open risk is that we've
          never load-tested checkout above 200 concurrent users.
        Sam: That worries me more than the date. Can we load test this week?
        Priya: We'd need a staging environment that mirrors production. We don't have one.
        Dan: I can build one, but it's about three days.
        Sam: Do it. We're approving a production-mirror staging environment for Atlas.
        Priya: Then 11 April is still fine.
        Sam: Also - pricing. Finance wants to raise the Pro tier from $19 to $24 at launch.
        Priya: That's going to affect the checkout copy and the tests.
        Sam: I know. Decision: we hold pricing at $19 for launch and revisit in June. I'm not
          bundling a price change into a rewrite launch.
        Dan: Sensible.
        """,
    },
    "M-04": {
        "project": "Beacon", "week": 12, "date": "2025-03-20", "title": "Beacon design review",
        "text": """
        Maya: Dan, walk us through the design doc.
        Dan: Short version - we reuse Atlas auth, add a new onboarding service, and the emails
          go through the shared vendor as agreed.
        Sam: Why a separate service? Why not put it in the Atlas monolith?
        Dan: Because Atlas is mid-rewrite. I don't want to add surface area to something that's
          about to launch.
        Sam: Fair. Approved - Beacon gets its own onboarding service.
        Maya: Contractor update: we've signed someone, they start 31 March.
        Sam: So the hiring risk is down.
        Maya: Mostly. We still need a permanent hire by June or the contractor extension gets
          expensive.
        Dan: One open question - do we need SOC2 review for the new service before launch?
        Maya: I'll find out. Let me own that and come back by 27 March.
        """,
    },
    "M-05": {
        "project": "Atlas", "week": 13, "date": "2025-03-26", "title": "Atlas weekly sync",
        "text": """
        Sam: Staging environment?
        Dan: Built. Load test ran yesterday.
        Priya: Results are not great. We fall over at about 350 concurrent users. Database
          connection pool is the bottleneck.
        Sam: Is that fixable before 11 April?
        Priya: Probably. It's a config change plus a connection pooler. Two days of work,
          then re-test.
        Sam: Then do it, and re-test before 4 April so we have a week of slack. Priya owns that.
        Priya: Understood.
        Dan: Do we tell the exec team about the load issue?
        Sam: Yes. We've decided to disclose the load test results to exec in Friday's update -
          I'd rather they hear it from us now than from customers in April.
        Priya: Agreed.
        """,
    },
    "M-06": {
        "project": "Beacon", "week": 14, "date": "2025-04-02", "title": "Beacon weekly sync",
        "text": """
        Maya: SOC2 - I have an answer. We do not need a full review before launch because the
          onboarding service doesn't store payment data. We do need a data flow diagram.
        Sam: Who does that?
        Maya: The contractor can, as part of onboarding. Good first task.
        Dan: Onboarding service is scaffolded, first activation email is sending in staging.
        Sam: Nice. Timeline still end of Q2?
        Maya: Yes, though I want to flag that we're now dependent on the Atlas auth team for
          one endpoint, and they're heads-down on launch.
        Sam: Noted, that's a real dependency risk. Decision: Beacon does not block on new Atlas
          auth work until after 11 April. Build against the existing endpoint.
        Dan: That means a small amount of rework later.
        Sam: Accepted. Dan, can you document the workaround by 9 April?
        """,
    },
}

for mid, m in TRANSCRIPTS.items():
    with open(f"transcripts/{mid}.txt", "w", encoding="utf-8") as f:   # "w" = write mode
        f.write(textwrap.dedent(m["text"]).strip())
        # textwrap.dedent removes the common leading indentation of our indented string
        # .strip() trims leading and trailing blank lines

print(f"wrote {len(TRANSCRIPTS)} transcripts:", sorted(os.listdir("transcripts")))
print("\nprojects:", sorted({m['project'] for m in TRANSCRIPTS.values()}))
print("weeks   :", sorted({m['week'] for m in TRANSCRIPTS.values()}))

## 2. LangChain — structured extraction  ⏱ 10 min

### The problem

The model can summarise a transcript into prose. Prose is useless to us — we need **data**: a list of decisions, a list of action items each with an owner and a due date.

Asking "reply in JSON" and calling `json.loads()` fails constantly: the model adds ```` ```json ```` fences, an apology, or a trailing comma.

### The solution

`llm.with_structured_output(Schema)` uses the provider's **tool-calling** machinery: your schema is sent as a function signature, the model fills it, and LangChain validates the result into a real Python object.

You describe the shape with **Pydantic**, and the `description=` texts are **sent to the model** — so the schema *is* part of your prompt.

📌 When asked *"how do you get reliable structured data out of an LLM?"* the answer is: *structured output via tool calling with a Pydantic schema, with `Field` descriptions acting as instructions, and nested models for repeated structures.*

In [ ]:
class ActionItem(BaseModel):
    """One thing one person agreed to do."""
    owner: str = Field(description="The single person who agreed to do it, first name as said.")
    task: str = Field(description="What they will do, in under 15 words, starting with a verb.")
    due: str = Field(description="The deadline exactly as stated (e.g. '21 March'), or 'not stated'.")

class MeetingNotes(BaseModel):
    """Structured notes extracted from one meeting transcript."""
    # ⚠️ This docstring is sent to the model as the tool description. Write it properly.

    summary: str = Field(description="Two sentences on what this meeting was about.")
    decisions: List[str] = Field(
        description=("Concrete decisions that were MADE, each a standalone sentence someone "
                     "who missed the meeting would understand. Exclude discussion and "
                     "suggestions - only things actually settled.")
    )
    action_items: List[ActionItem] = Field(
        description="Commitments made by a named person. Empty list if none."
    )
    risks: List[str] = Field(
        description="Risks or blockers raised, each one short sentence. Empty list if none."
    )

# .with_structured_output() returns a NEW Runnable. The original `llm` is unchanged,
# so you can keep several bound variants side by side.
extractor = llm.with_structured_output(MeetingNotes)

EXTRACT_PROMPT = """Extract structured notes from this meeting transcript.

Be strict about the difference between a DECISION and a discussion:
- "Let's cut Apple Pay from v1"           -> a decision
- "Should we support Apple Pay?"          -> not a decision
- "I'd want a decision on the vendor"     -> not a decision

Meeting: {title} ({project}, week {week}, {date})

TRANSCRIPT:
{text}"""

# Test on ONE transcript first. Always validate a single case before running a batch -
# it is much cheaper to fix a prompt on one example than on sixty.
m = TRANSCRIPTS["M-01"]
# ** unpacks the dict into keyword arguments. Note we do NOT also pass text= :
# the dict already HAS a "text" key, and passing it twice raises
#   TypeError: str.format() got multiple values for keyword argument 'text'
notes = extractor.invoke(EXTRACT_PROMPT.format(**m))

print("type:", type(notes).__name__)
print("\nSUMMARY:", notes.summary)
print("\nDECISIONS:")
for d in notes.decisions:
    print("  -", d)
print("\nACTION ITEMS:")
for a in notes.action_items:
    print(f"  - [{a.owner}] {a.task}  (due: {a.due})")
print("\nRISKS:")
for r in notes.risks:
    print("  -", r)

### Line by line: why the schema looks like that 📌

| Choice | Why |
|---|---|
| `class ActionItem(BaseModel)` nested inside `MeetingNotes` | an action item is three fields that belong together. Nested models are how you express repeated structure — far better than three parallel lists you have to zip back up |
| `owner: str` (not `List[str]`) | forces one accountable person. "Priya and Dan will…" is not an action item, it's two |
| `due: str` with `"not stated"` | dates in transcripts are vague ("before Friday"). A `str` keeps what was actually said. Forcing a `date` type would make the model **invent** a specific date |
| `decisions: List[str]` with a long `description` | the description carries three worked examples of decision-vs-discussion. **That is prompt engineering inside the schema** |
| `Field(description=…)` on every field | all of these are sent to the model. An undescribed field is an unprompted field |
| `"Empty list if none"` | without it, models pad empty lists with plausible-sounding invented items |

In [ ]:
# ---------- run the extraction over all six transcripts ----------
# .batch() runs the calls CONCURRENTLY. Six meetings take roughly the time of one,
# not six. Most people write a for-loop here and wait six times as long.
prompts = [EXTRACT_PROMPT.format(**m) for m in TRANSCRIPTS.values()]

t0 = time.time()
all_notes = extractor.batch(prompts)                    # -> list[MeetingNotes]
print(f"extracted {len(all_notes)} meetings in {time.time()-t0:.1f}s\n")

# zip pairs each meeting id with its notes, in the same order we sent them.
notes_by_id = dict(zip(TRANSCRIPTS.keys(), all_notes))

for mid, n in notes_by_id.items():
    m = TRANSCRIPTS[mid]
    print(f"{mid} | {m['project']:<7} w{m['week']} | "
          f"{len(n.decisions)} decisions, {len(n.action_items)} actions, {len(n.risks)} risks")

# To cap parallelism and respect rate limits on a bigger batch:
# all_notes = extractor.batch(prompts, config={"max_concurrency": 5})

## 3. The action register  ⏱ 5 min

This is the **deliverable** — the thing you could paste into a stand-up on Monday morning.

It is also the payoff of structured output: once the data is typed, everything downstream is **ordinary programming**. No AI involved in this cell at all.

In [ ]:
import pandas as pd

rows = []
for mid, n in notes_by_id.items():
    m = TRANSCRIPTS[mid]
    for a in n.action_items:
        rows.append({
            "owner": a.owner,
            "task": a.task,
            "due": a.due,
            "project": m["project"],
            "week": m["week"],
            "meeting": mid,
            "date": m["date"],
        })

register = pd.DataFrame(rows)

print("=" * 78)
print("ACTION REGISTER")
print("=" * 78)
if len(register):
    # .sort_values sorts by column; to_string(index=False) hides the row numbers
    print(register.sort_values(["owner", "week"])[
        ["owner", "task", "due", "project", "meeting"]].to_string(index=False))

    print("\nitems per owner:", dict(register["owner"].value_counts()))
    print("items per project:", dict(register["project"].value_counts()))
    # A genuinely useful flag: commitments with no deadline are the ones that rot.
    no_due = register[register["due"].str.contains("not stated", case=False, na=False)]
    print(f"\n⚠️  {len(no_due)} action item(s) have no stated deadline - chase these first")
else:
    print("(no action items extracted - re-run the extraction cell)")

## 4. ChromaDB — persistent, filterable memory  ⏱ 15 min

### What we index, and why it matters

We do **not** index whole transcripts. We index each **decision, action item and risk as its own `Document`**.

That choice is the whole trick:

| If you index… | Then a search returns… |
|---|---|
| whole transcripts | a 400-word transcript, and the model has to find the decision inside it |
| **one decision per document** | **the decision itself**, already isolated, already tagged with project, week and owner |

We already did the hard work in §2 — the extraction produced clean atomic facts. Indexing them individually turns that into precise retrieval.

### Chroma metadata rules

- Keep values to **`str`, `int`, `float`, `bool`**. Scalars filter reliably across versions; lists are not portable.
- The **collection name** must be 3–512 characters from `[a-zA-Z0-9._-]`. `"m1"` fails; `"meeting_memory"` works. (Yes, this is a real error people hit.)

In [ ]:
from langchain_chroma import Chroma

chroma_docs = []

for mid, n in notes_by_id.items():
    m = TRANSCRIPTS[mid]

    # Metadata shared by everything from this meeting.
    base = {"meeting": mid, "project": m["project"], "week": int(m["week"]),
            "date": m["date"], "title": m["title"]}

    for d in n.decisions:
        chroma_docs.append(Document(
            page_content=d,                                   # the searchable text
            metadata={**base, "kind": "decision", "owner": "team"},
        ))   # {**base, ...} copies base and adds keys - a clean way to extend a dict

    for a in n.action_items:
        chroma_docs.append(Document(
            page_content=f"{a.task} (owner: {a.owner}, due: {a.due})",
            metadata={**base, "kind": "action", "owner": a.owner, "due": a.due},
        ))

    for r in n.risks:
        chroma_docs.append(Document(
            page_content=r,
            metadata={**base, "kind": "risk", "owner": "team"},
        ))

# ⚠️ Chroma raises on an empty list, so guard it. Defensive, and it also tells you
# clearly that the problem is upstream in extraction, not here.
if not chroma_docs:
    chroma_docs = [Document(page_content="No notes were extracted.",
                            metadata={"meeting": "none", "project": "none", "week": 0,
                                      "date": "", "title": "", "kind": "note", "owner": "team"})]

print(f"{len(chroma_docs)} documents to index")
print("by kind:", dict(pd.Series([d.metadata["kind"] for d in chroma_docs]).value_counts()))
print("\nexample decision document:")
ex = next(d for d in chroma_docs if d.metadata["kind"] == "decision")
print("  text    :", ex.page_content)
print("  metadata:", json.dumps(ex.metadata))

In [ ]:
# ---------- build the Chroma collection ----------
import shutil
shutil.rmtree(CHROMA_DIR, ignore_errors=True)     # start clean so re-running this cell
                                                   # doesn't create duplicate entries

t0 = time.time()
memory = Chroma.from_documents(
    documents=chroma_docs,
    embedding=emb,                      # the SAME embedding model must be used for
                                        #   indexing AND querying - different models
                                        #   produce incompatible vector spaces
    collection_name=COLLECTION,         # 3+ characters, see the note above
    persist_directory=CHROMA_DIR,       # ⭐ this one argument makes it persistent.
                                        #   No save_local() call needed - it's on disk.
)
print(f"indexed {memory._collection.count()} documents in {time.time()-t0:.1f}s")
print("files on disk:", os.listdir(CHROMA_DIR)[:5])

# Prove persistence: open the SAME collection again as if this were a fresh process.
# In a real app this is what your server does on startup - and it costs $0, because
# nothing is re-embedded.
reopened = Chroma(
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
    embedding_function=emb,             # note: the argument is embedding_function here,
)                                       #   but `embedding` in from_documents. Easy to trip on.
print("\nreopened from disk ->", reopened._collection.count(), "documents")

In [ ]:
# ---------- QUERY STYLE 1: semantic search (meaning) ----------
# Note that none of these queries use the exact words in the decisions.
for q in ["when are we launching?",
          "did we change any prices?",
          "what are we doing about performance problems?"]:
    print(f"\nQ: {q}")
    for d in memory.similarity_search(q, k=2):
        mt = d.metadata
        print(f"   [{mt['kind']}] {mt['project']} w{mt['week']}: {d.page_content[:80]}")

In [ ]:
# ---------- QUERY STYLE 2: semantic search + metadata filter ----------
# Chroma's filter operators: $eq  $ne  $gt  $gte  $lt  $lte  $in  $nin  $and  $or
print("Q: 'what did we decide?' restricted to Atlas, week 12 or later\n")

hits = memory.similarity_search(
    "what did we decide?",
    k=4,
    filter={"$and": [                       # $and takes a LIST of conditions
        {"project": "Atlas"},               # bare value = equality (shorthand for $eq)
        {"kind": "decision"},
        {"week": {"$gte": 12}},             # numeric comparison - needs the operator form
    ]},
)
for d in hits:
    print(f"   w{d.metadata['week']}: {d.page_content[:90]}")

print("\n📌 Filtering happens INSIDE the query, so out-of-scope documents are never")
print("   candidates. This is also how you build multi-tenant systems: put the")
print("   tenant id in metadata and filter on it. Never rely on the prompt to keep")
print("   one customer's data away from another's.")

In [ ]:
# ---------- QUERY STYLE 3: metadata ONLY - no embeddings at all ----------
# ⭐ THE KEY CELL OF THIS SECTION.
# "What does Priya owe?" is not a similarity question. It is a filter.
# Chroma's .get() answers it WITHOUT calling the embedding model: no vector maths,
# no API call, no cost, and - crucially - no chance of missing an item because it
# happened to be semantically further away.

def items_for_owner(owner: str) -> List[dict]:
    """Every action item owned by one person. Exact, complete, and free."""
    got = memory.get(                        # .get() = metadata query, not a search
        where={"$and": [{"kind": "action"}, {"owner": owner}]},
        include=["documents", "metadatas"],  # what to return
    )
    # .get() returns parallel lists, not Documents. zip stitches them back together.
    return [{"text": t, **md} for t, md in zip(got["documents"], got["metadatas"])]

print("Priya's action items (metadata query, zero embedding calls):")
for it in items_for_owner("Priya"):
    print(f"   [{it['meeting']} w{it['week']}] {it['text']}")

print("\nCompare with vector search for the same question:")
for d in memory.similarity_search("what does Priya owe?", k=3):
    print(f"   [{d.metadata['kind']}, owner={d.metadata['owner']}] {d.page_content[:70]}")

print("\n📌 THE LESSON: vector search returns what is SIMILAR, not what MATCHES.")
print("   For 'all items where owner = X', use a filter. Getting this wrong gives")
print("   you an incomplete list that looks perfectly plausible - the worst failure.")

## 5. LangGraph — state and the router  ⏱ 10 min

### Why we need it

Two things a plain LangChain chain cannot do, both of which this project needs:

1. **Choose between different pipelines.** A chain is a fixed line. We need *"is this a meaning question, a filter question, or small talk?"* and then three different behaviours.
2. **Remember across turns.** The user asks *"what did we decide about pricing?"*, then *"who raised that?"*. The second message alone is meaningless.

### The four primitives — this is all of LangGraph 📌

| Primitive | What it is | Here |
|---|---|---|
| **State** | a shared dict, typed with `TypedDict`, flowing through the graph | messages, route, context |
| **Node** | a plain Python function: state in, **partial** update out | our 4 nodes |
| **Edge** | "after A, always go to B" | `search → answer` |
| **Conditional edge** | a function that reads state and *chooses* the next node | the 3-way route |

Plus the **checkpointer**: it saves state after every node, keyed by a `thread_id` — which is what gives memory, crash recovery and per-user isolation.

In [ ]:
class MindState(TypedDict):
    # Annotated[list, add_messages] attaches a REDUCER.
    # A reducer decides how a node's update merges with the existing value:
    #   - the DEFAULT reducer OVERWRITES the field
    #   - add_messages APPENDS, and coerces raw dicts into message objects
    # For a conversation you want append. This single line is the memory mechanism. ⭐
    messages: Annotated[list, add_messages]

    route: str        # written by classify: "recall" | "owner" | "chitchat"
    owner: str        # written by classify: whose items, if this is an owner question
    context: str      # written by the recall/owner nodes: what the answer must be based on

class Route(BaseModel):
    """Which pipeline should handle this message?"""
    kind: Literal["recall", "owner", "chitchat"] = Field(
        description=("'recall' = asks what was decided, discussed, or what the risks are. "
                     "'owner' = asks what a SPECIFIC PERSON is responsible for, or what is "
                     "assigned to them. 'chitchat' = greeting, thanks, or a question about "
                     "the assistant itself.")
    )
    owner: Optional[str] = Field(
        default=None,
        description="If kind is 'owner', the person's first name as written. Else null."
    )
    # Optional with default=None is the correct way to say "may be absent".
    # Without it the model INVENTS a name to satisfy the schema - a very common bug.

router = llm.with_structured_output(Route)

def classify_node(state: MindState) -> dict:
    last = state["messages"][-1].content        # the newest message
    r = router.invoke(f"Classify this message for a meeting-notes assistant.\n\nMessage: {last}")
    print(f"   [classify] {r.kind}" + (f" (owner={r.owner})" if r.owner else ""))
    return {"route": r.kind, "owner": r.owner or ""}     # a PARTIAL update

print("state + classify node defined")

### Why routing works reliably here 📌

Because `Route.kind` is a `Literal`, the value **cannot** be `"Recall."` or `"i think owner"`. Tool calling constrains it to one of three exact strings, and Pydantic validates it.

That is the difference between a router that works and one that breaks every fiftieth request on a stray full stop. **Reliable routing depends on reliable classification** — which is why structured output came first.

In [ ]:
# ---------- NODE 2: recall (semantic search) ----------
def recall_node(state: MindState) -> dict:
    """Meaning questions -> vector search over decisions, risks and actions."""
    question = state["messages"][-1].content

    # Include a little history so follow-ups ("who raised that?") search sensibly.
    history = " ".join(m.content for m in state["messages"][-4:-1])
    query = f"{history} {question}".strip() if history else question

    docs = memory.similarity_search(query, k=5)
    print(f"   [recall] {len(docs)} notes retrieved")

    # Label each result with its source, so the model can cite it. Without this the
    # model has no idea where the text came from and invents citations.
    lines = [f"- [{d.metadata['kind']} | {d.metadata['project']} week "
             f"{d.metadata['week']} | {d.metadata['meeting']}] {d.page_content}"
             for d in docs]
    return {"context": "\n".join(lines) if lines else "(nothing found)"}

# ---------- NODE 3: owner (metadata filter) ----------
def owner_node(state: MindState) -> dict:
    """Filter questions -> exact metadata query. No embeddings, no similarity."""
    owner = state.get("owner", "").strip()
    if not owner:
        return {"context": "(no person was named in the question)"}

    items = items_for_owner(owner)
    print(f"   [owner] {len(items)} items for {owner}")

    if not items:
        return {"context": f"(no action items are recorded for {owner})"}
    lines = [f"- [{it['project']} week {it['week']} | {it['meeting']}] {it['text']}"
             for it in items]
    return {"context": "\n".join(lines)}

print("recall + owner nodes defined")

In [ ]:
# ---------- NODE 4: answer ----------
# One grounded prompt serves both paths, because both produce a `context` block.
ANSWER_SYSTEM = """You are MeetingMind, an assistant over one team's meeting notes.

Rules:
1. Answer using ONLY the NOTES below. Never use outside knowledge or guess.
2. If the notes do not contain the answer, reply exactly:
   "That isn't in the meeting notes I have."
3. Cite the meeting for every fact, like this: [M-03, week 12].
4. Be concise: at most 4 sentences, or a short bullet list for multiple items.

NOTES:
{context}"""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", ANSWER_SYSTEM),
    MessagesPlaceholder("history"),      # inserts a LIST of past messages - the memory
    ("human", "{question}"),
])

def answer_node(state: MindState) -> dict:
    msg = (answer_prompt | llm).invoke(
        {
            "context": state["context"],
            "history": state["messages"][:-1],       # everything except the new question
            "question": state["messages"][-1].content,
        },
        # This config is what LangSmith uses to label the run. It costs nothing when
        # tracing is off, and makes traces searchable when it is on.
        config={"run_name": "meetingmind_answer",
                "tags": ["meetingmind", state.get("route", "?")]},
    )
    return {"messages": [msg]}                       # add_messages APPENDS this

# ---------- NODE 5: chitchat ----------
def chitchat_node(state: MindState) -> dict:
    """No search, no filter. Saves a retrieval and a wasted grounded call on 'thanks!'."""
    msg = llm.invoke([
        SystemMessage("You are MeetingMind, an assistant over a team's meeting notes. "
                      "Reply in ONE short sentence and invite a question about decisions, "
                      "risks or action items."),
        *state["messages"][-4:],                     # * unpacks the list as arguments
    ])
    return {"messages": [msg]}

print("answer + chitchat nodes defined")

In [ ]:
# ---------- WIRE IT UP ----------
# The build sequence is identical in every LangGraph project:
#   StateGraph -> add_node -> add_edge -> compile -> invoke
def route_decider(state: MindState) -> str:
    """The routing function: reads state, returns the NAME of the next node."""
    # A dict lookup with a DEFAULT. A safe fallback branch is mandatory - never let an
    # unexpected value crash the graph.
    return {"recall": "recall", "owner": "owner"}.get(state["route"], "chitchat")

b = StateGraph(MindState)                       # needs the state class to validate updates

b.add_node("classify", classify_node)           # register each node under a NAME;
b.add_node("recall", recall_node)               #   edges refer to these strings
b.add_node("owner", owner_node)
b.add_node("answer", answer_node)
b.add_node("chitchat", chitchat_node)

b.add_edge(START, "classify")                   # START = the virtual entry point
b.add_conditional_edges(                        # ⭐ THE 3-WAY BRANCH
    "classify",                                 #   after this node...
    route_decider,                              #   ...call this to decide...
    ["recall", "owner", "chitchat"],            #   ...and it returns one of these
)                                               #   (the list drives validation + diagrams)

# Both data paths converge on the SAME answer node - they only differ in HOW they
# filled `context`. That is a nice property: one grounded prompt, two retrieval styles.
b.add_edge("recall", "answer")
b.add_edge("owner", "answer")
b.add_edge("answer", END)                       # END = the virtual exit
b.add_edge("chitchat", END)

# ⭐ THE CHECKPOINTER saves state after every node, keyed by thread_id.
# InMemorySaver is literally a dict - fine for a notebook.
# In production: PostgresSaver, and NO other code changes.
mind = b.compile(checkpointer=InMemorySaver())

# draw_mermaid() emits Mermaid text, which GitHub renders natively. Paste it into a
# README and you have an architecture diagram that cannot drift from the code.
print(mind.get_graph().draw_mermaid())

## 6. Running the assistant  ⏱ 5 min

A "thread" is one conversation. `thread_id` is how you say which one — in a real app it is the user id or session id.

Notice we send **only the new message** each time. The checkpointer reloads the history.

In [ ]:
def ask(question: str, thread: str = "demo") -> dict:
    """Send one message to MeetingMind and print the reply."""
    cfg = {"configurable": {"thread_id": thread}}
    print(f"\nUSER: {question}")
    out = mind.invoke({"messages": [HumanMessage(question)]}, cfg)
    #                   ^ ONLY the new message; history comes from the checkpointer
    print(f"BOT : {out['messages'][-1].content}")
    return out

# ---------- PATH A: a meaning question ----------
r1 = ask("What did we decide about the Atlas launch date, and why?")

In [ ]:
# ---------- PATH A as a FOLLOW-UP ----------
# "What about pricing?" is meaningless alone. The recall node folds recent history
# into the search query, so it stays on the Atlas thread.
r2 = ask("and what about pricing?")

In [ ]:
# ---------- PATH B: an owner question (metadata filter, no embeddings) ----------
r3 = ask("What is Priya on the hook for?")

print("\nroute taken:", r3.get("route"), "| owner detected:", r3.get("owner"))
print("\nThis answer came from a metadata query, so it is COMPLETE - every item where")
print("owner = Priya, not just the ones that happened to be semantically nearest.")

In [ ]:
# ---------- more of both paths ----------
r4 = ask("What are the biggest risks on Beacon?")
r5 = ask("What does Dan need to do?")

In [ ]:
# ---------- PATH C: small talk, and the refusal test ----------
r6 = ask("thanks, this is really useful!")
print("   route:", r6.get("route"), "-> no search, no filter, no grounded call")

# THE MOST IMPORTANT TEST: ask about something never discussed.
# A good assistant declines. A bad one invents a decision that was never made -
# which, in a tool people trust for "what did we agree?", is genuinely damaging.
r7 = ask("What did we decide about the office move to Berlin?", thread="refusal-test")

text = r7["messages"][-1].content.lower()
refused = "isn't in the meeting notes" in text or "not in the meeting notes" in text
print("\n" + ("PASS - it declined instead of inventing a decision" if refused
              else "REVIEW - check whether it invented something"))

In [ ]:
# ---------- memory is real, and threads are isolated ----------
snap = mind.get_state({"configurable": {"thread_id": "demo"}})
print(f"'demo' thread holds {len(snap.values['messages'])} messages")
print(f"next node to run: {snap.next}   (empty tuple = the run finished)")

# get_state is the single most useful debugging call in LangGraph: it shows exactly
# what was saved and where the graph is.
other = ask("what did I ask you first?", thread="someone-else")
print("\n^ a different thread_id has no history, so it cannot know. That is user isolation,")
print("  and it is why thread_id should be your user or session id in production.")

## 7. LangSmith — tracing  ⏱ 2 min

Setup is **three environment variables and zero code changes** — LangChain and LangGraph emit trace data automatically through their callback system.

| What you get | Why it matters here |
|---|---|
| the **exact prompt** sent | "why did it cite M-03?" is answered instantly |
| per-step latency | is the slow part the classifier, the search, or the answer? |
| tokens and cost per step | extraction is your expensive step — now you can see it |
| filter by tag | `tags=["meetingmind", route]` lets you compare the two paths |

Free key at `smith.langchain.com` → Settings → API keys. **Leave it blank and everything below still runs** — §8 is entirely local.

In [ ]:
# Paste a LangSmith key to enable the hosted dashboard, or leave it empty.
LANGSMITH_KEY = ""        # e.g. "lsv2_pt_..."

if LANGSMITH_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_KEY
    os.environ["LANGSMITH_TRACING"] = "true"          # the master switch
    os.environ["LANGSMITH_PROJECT"] = "meetingmind"
    print("Tracing ON -> project 'meetingmind'. Open https://smith.langchain.com")
    print("\nEach run appears as a tree:")
    print("  meetingmind")
    print("    ├── classify  -> Route(kind='owner', owner='Priya')")
    print("    ├── owner     -> the metadata query")
    print("    └── answer    -> THE EXACT PROMPT, tokens, cost, latency")
else:
    os.environ["LANGSMITH_TRACING"] = "false"         # explicitly off so nothing errors
    print("Tracing OFF (no key). Everything below still RUNS.")

LANGSMITH_ON = bool(LANGSMITH_KEY)

# @traceable puts YOUR plain Python into the same trace tree as the LangChain parts.
# Our metadata query is normal code, so LangChain knows nothing about it - until now.
@traceable(run_type="retriever", name="chroma_owner_lookup")
def items_for_owner_traced(owner: str):
    return items_for_owner(owner)

print("\ntraced lookup ->", len(items_for_owner_traced("Dan")), "items for Dan")
print("(the decorator is a no-op when tracing is off, so leave it in your code)")

## 8. LangSmith — four metrics and a scorecard  ⏱ 5 min

### Evaluation theory 📌

This app has **three** places it can fail, and they need **different** metrics:

```
            ROUTING                  RETRIEVAL                 GENERATION
               │                         │                          │
message ─► which pipeline? ─► did we find the right notes? ─► is the answer honest?
               │                         │                          │
      routing accuracy          owner-list completeness      faithfulness, refusal
           (free)                      (free)                     (judge)
```

**Measure the free ones first.** Routing accuracy and completeness cost nothing, run instantly, and are usually where the bugs are.

| Metric | What it catches | Cost |
|---|---|---|
| **Routing accuracy** | an owner question sent down the vector-search path | free |
| **Owner completeness** | an incomplete list of someone's commitments | free |
| **Faithfulness** | citing a meeting that doesn't support the claim | 1 judge call |
| **Refusal accuracy** | inventing a decision that was never made | free |

### The golden set rules

1. **10–30 cases** is plenty to start.
2. Write them from **real phrasings**, not by reading your schema.
3. **Always include cases the data cannot answer.** If everything is answerable, you are not testing the refusal path — the most dangerous failure mode.
4. Cover **every route**.

In [ ]:
GOLDEN = [
    # ---- recall (meaning) ----
    {"msg": "What did we decide about the Atlas launch date?", "route": "recall",
     "must_mention": "11 April"},
    {"msg": "Did we change the pricing?", "route": "recall", "must_mention": "19"},
    {"msg": "What are the risks on Beacon?", "route": "recall"},
    {"msg": "What did we decide about SOC2?", "route": "recall"},
    {"msg": "Why did Beacon get its own service?", "route": "recall"},

    # ---- owner (filter) ----
    {"msg": "What is Priya on the hook for?", "route": "owner", "owner": "Priya"},
    {"msg": "What does Dan need to do?", "route": "owner", "owner": "Dan"},
    {"msg": "What are Maya's action items?", "route": "owner", "owner": "Maya"},

    # ---- chitchat ----
    {"msg": "hi there", "route": "chitchat"},
    {"msg": "thanks, that helps", "route": "chitchat"},

    # ---- must REFUSE: never discussed in any meeting ----
    {"msg": "What did we decide about the Berlin office move?", "route": "recall",
     "must_refuse": True},
    {"msg": "What is our 2027 revenue target?", "route": "recall", "must_refuse": True},
]

print(f"{len(GOLDEN)} golden cases")
print("routes:", dict(pd.Series([c["route"] for c in GOLDEN]).value_counts()))
print("refusal cases:", sum(1 for c in GOLDEN if c.get("must_refuse")))

In [ ]:
# ---------- THE EVALUATORS ----------
# An evaluator is just a function that scores an output. No framework magic; the
# {"key": ..., "score": ...} convention is simply what makes results chartable.

class Verdict(BaseModel):
    """A judge's binary verdict with a one-line reason."""
    passed: bool = Field(description="True if the criterion is met.")
    why: str = Field(description="One short sentence explaining the verdict.")

judge = llm.with_structured_output(Verdict)     # a cheap model is fine: judging is
                                                # easier than generating

def eval_faithful(answer: str, context: str) -> float:
    """Is every claim in the answer supported by the notes the app actually retrieved?

    The anti-hallucination metric. Note it needs NO reference answer - it compares
    the answer against the context, which is what makes it cheap to maintain.
    """
    v = judge.invoke(
        f"NOTES GIVEN TO THE ASSISTANT:\n{context}\n\nASSISTANT ANSWER:\n{answer}\n\n"
        "passed=true if every factual claim in the answer is supported by the notes, "
        "OR if the answer declines to answer. passed=false if it states anything the "
        "notes do not support."
    )
    return float(v.passed)

REFUSAL_PHRASES = ["isn't in the meeting notes", "not in the meeting notes",
                   "isn't in the notes", "don't have"]

def eval_refusal(answer: str) -> float:
    """Plain string check. Prefer code over a judge whenever code will do the job."""
    return 1.0 if any(p in answer.lower() for p in REFUSAL_PHRASES) else 0.0

def eval_owner_complete(answer: str, owner: str) -> Optional[float]:
    """Did the answer include EVERY item that person owns?

    We know the truth from our own register, so this is exact - no judge needed.
    Deterministic metrics are always better than judged ones when available.
    """
    if len(register) == 0:
        return None
    expected = register[register["owner"].str.lower() == owner.lower()]
    if len(expected) == 0:
        return None
    # A crude but effective check: does the answer mention a distinctive word from
    # each task? Crude beats absent - and it is free and never flakes.
    found = 0
    for task in expected["task"]:
        keywords = [w for w in task.lower().split() if len(w) > 5]
        if any(kw.strip(".,()") in answer.lower() for kw in keywords[:3]):
            found += 1
    return round(found / len(expected), 2)

# Prove the judge catches an unsupported claim - test your tests.
ctx = "- [decision | Atlas week 11 | M-01] Atlas launch moves to 11 April."
print("supported  ->", eval_faithful("The launch moved to 11 April [M-01, week 11].", ctx))
print("unsupported->", eval_faithful("The launch moved to 11 April and Sam resigned.", ctx))

In [ ]:
# ---------- RUN THE EVALUATION ----------
def evaluate(label: str = "baseline", verbose: bool = True) -> dict:
    """Run every golden case through the WHOLE graph and score it. No account needed."""
    started = time.time()
    scores = {"routing": [], "faithful": [], "refusal": [], "owner_complete": [], "mention": []}
    rows = []

    for i, case in enumerate(GOLDEN):
        # A FRESH thread per case, so one answer cannot leak into the next.
        cfg = {"configurable": {"thread_id": f"eval-{label}-{i}"}}
        out = mind.invoke({"messages": [HumanMessage(case["msg"])]}, cfg)
        answer = out["messages"][-1].content
        context = out.get("context", "")

        # --- metric 1: routing accuracy (free) ---
        actual = out.get("route", "?")
        routing = 1.0 if actual == case["route"] else 0.0
        scores["routing"].append(routing)

        # --- metric 2: faithfulness (judge; only where there IS context) ---
        faith = None
        if context and not case.get("must_refuse"):
            faith = eval_faithful(answer, context)
            scores["faithful"].append(faith)

        # --- metric 3: refusal (free) ---
        refuse = None
        if case.get("must_refuse"):
            refuse = eval_refusal(answer)
            scores["refusal"].append(refuse)

        # --- metric 4: owner completeness (free, exact) ---
        complete = None
        if case.get("owner"):
            complete = eval_owner_complete(answer, case["owner"])
            if complete is not None:
                scores["owner_complete"].append(complete)

        # --- bonus: does the answer contain the fact we know it should? (free) ---
        mention = None
        if case.get("must_mention"):
            mention = 1.0 if case["must_mention"].lower() in answer.lower() else 0.0
            scores["mention"].append(mention)

        def mark(s):
            return "-" if s is None else ("PASS" if s >= 1 else ("FAIL" if s == 0 else f"{s:.2f}"))
        rows.append((mark(routing), mark(faith), mark(refuse), mark(complete),
                     actual[:8], case["msg"][:38]))

    if verbose:
        print(f"{'ROUTE':<7}{'FAITH':<7}{'REFUSE':<8}{'COMPL':<8}{'WENT TO':<10}MESSAGE")
        for r in rows:
            print(f"{r[0]:<7}{r[1]:<7}{r[2]:<8}{r[3]:<8}{r[4]:<10}{r[5]}")

    # Average each metric, skipping any with no applicable cases.
    result = {m: round(sum(v) / len(v), 2) for m, v in scores.items() if v}
    result["seconds"] = round(time.time() - started, 1)
    if verbose:
        print(f"\nSCORECARD [{label}]: {result}")
    return result

scorecard = evaluate(label="v1")

### Reading the scorecard 📌

Each metric points at a **different** fix. That is exactly why we measure them separately.

| What you see | What it means | What to change |
|---|---|---|
| routing low | the classifier confuses "what does Dan need to do" with a recall question | sharpen the `Field` descriptions in `Route`; add few-shot examples |
| owner completeness low | the metadata filter is missing items | check the `owner` string matches exactly (case, nicknames) — normalise names at extraction |
| faithfulness low | it is adding claims the notes don't support | strengthen rules 1 and 3 in `ANSWER_SYSTEM` |
| refusal low | it invents decisions | give it an exact sentence to output; keep `temperature=0` |
| everything high, users unhappy | **your golden set doesn't look like real questions** | log real messages and grow the set from failures |

### The method for improving anything, forever

1. change **exactly one thing**
2. re-run the **same** golden set
3. keep the change only if the number went up

Try it: delete rule 2 from `ANSWER_SYSTEM`, re-run `evaluate(label="no-refusal-rule")`, and watch refusal accuracy collapse. Then put it back. That is a five-minute experiment that teaches more than an hour of reading.

## 9. Self-check, next steps, and the resume bullet

In [ ]:
checks = []
def ck(name, condition):
    checks.append((name, bool(condition)))

# --- data + extraction ---
ck("6 transcripts written", len(TRANSCRIPTS) == 6)
ck("extraction returned typed objects", all(isinstance(n, MeetingNotes) for n in all_notes))
ck("notes for every meeting", len(notes_by_id) == 6)
ck("action register is a dataframe", isinstance(register, pd.DataFrame))

# --- Chroma ---
ck("chroma indexed the documents", memory._collection.count() == len(chroma_docs))
ck("chroma persisted to disk", os.path.isdir(CHROMA_DIR) and len(os.listdir(CHROMA_DIR)) > 0)
ck("reopened collection matches", reopened._collection.count() == memory._collection.count())
ck("semantic search returns results", len(memory.similarity_search("launch date", k=2)) > 0)
ck("metadata filter works",
   all(d.metadata["project"] == "Atlas"
       for d in memory.similarity_search("decision", k=3, filter={"project": "Atlas"})))
ck("metadata-only get works", isinstance(items_for_owner("Priya"), list))

# --- LangGraph ---
ck("graph has 5 nodes",
   len([n for n in mind.get_graph().nodes if not n.startswith("__")]) == 5)
ck("recall path produced context", bool(r1.get("context")))
ck("owner path was routed", r3.get("route") in ("owner", "recall", "chitchat"))
ck("memory persisted the thread",
   len(mind.get_state({"configurable": {"thread_id": "demo"}}).values["messages"]) >= 4)
ck("threads are isolated",
   len(mind.get_state({"configurable": {"thread_id": "someone-else"}}).values["messages"]) <= 2)

# --- evaluation ---
ck("golden set covers all 3 routes", len({c["route"] for c in GOLDEN}) == 3)
ck("golden set has refusal cases", sum(1 for c in GOLDEN if c.get("must_refuse")) >= 2)
ck("scorecard produced", "routing" in scorecard)

for name, ok in checks:
    print(f"{'ok  ' if ok else 'FAIL'} {name}")

failed = [n for n, ok in checks if not ok]
print("\n" + ("ALL CHECKS PASSED" if not failed else f"FAILED: {failed}"))

### What you built in an hour

```
LangChain   ──►  nested-Pydantic extraction (decisions, owners, due dates, risks)
                 · batch() for concurrency · one grounded prompt with a refusal path

ChromaDB    ──►  persistent collection · atomic notes as documents
                 · semantic search  ·  $and/$gte/$in filters
                 ·  .get(where=…) for exact metadata queries with NO embeddings

LangGraph   ──►  5 nodes, a 3-way conditional edge, two retrieval styles converging
                 on one answer node · checkpointer for memory and user isolation

LangSmith   ──►  tracing config on every answer · @traceable over the Chroma lookup
                 · 12-case golden set · 4 metrics · a scorecard
```

### The five ideas worth keeping

1. **Vector search returns what is *similar*; a filter returns what *matches*.** "All items owned by X" is a filter. Getting this wrong gives an incomplete list that looks plausible — the worst kind of failure.
2. **Index atomic facts, not whole documents.** Extraction did the hard work; one decision per document turns that into precise retrieval.
3. **Structured output is the glue.** Once the data is typed, the action register is ordinary pandas.
4. **The refusal path is a feature.** A tool people trust for "what did we agree?" must never invent an agreement.
5. **Free metrics first.** Routing accuracy and completeness cost nothing and are where the bugs live.

### Next steps, in order of value

| Effort | Do this |
|---|---|
| 10 min | Replace the transcripts with **your own** meeting notes or Slack exports. Everything else works unchanged. |
| 15 min | Add a `"stats"` route: *"how many open actions per person?"* — a pandas query over `register`, same pattern as `owner`. |
| 20 min | Normalise owner names at extraction (`"priya"`, `"Priya S."` → `"Priya"`) and watch owner-completeness rise. |
| 20 min | Swap `InMemorySaver` → `PostgresSaver` and prove the conversation survives a restart. |
| 30 min | Wrap it in Streamlit: paste a transcript, get notes, then chat. `pip install streamlit`, ~60 lines. |
| 30 min | Add hybrid search (BM25 + vector) so exact project codes and names always match. |

### Resume bullet (replace with your own measured numbers)

> **MeetingMind — Meeting Intelligence Assistant** · *Python · LangChain · LangGraph · ChromaDB · LangSmith · OpenAI*
> - Built an assistant that extracts decisions, owner-tagged action items and risks from raw meeting transcripts using nested Pydantic structured output, producing an action register with deadline gaps flagged automatically.
> - Indexed each decision and action as an atomic document in a persistent ChromaDB collection with project, week and owner metadata, enabling both semantic recall and exact filtered lookups.
> - Recognised that "what is X responsible for?" is a filter rather than a similarity query, and routed it to a Chroma metadata query (`.get(where=…)`) that needs no embedding call — turning an incomplete plausible list into a complete exact one.
> - Implemented the flow as a LangGraph state machine with a three-way conditional edge and a checkpointer keyed by `thread_id`, giving working multi-turn follow-ups and per-user isolation.
> - Instrumented with LangSmith tracing and a 12-case golden set scoring routing accuracy, owner-list completeness, faithfulness and refusal rate.